# 05 — Neutral Income Lab: Ranges, Volatility, and Gamma

You will build the straddle/strangle family, the iron condor, iron butterfly, and long butterfly;
summarize each; overlay the **expected move** on an iron condor; and stress a **short strangle**
across price and time with a scenario grid + P&L heatmap to *see* gamma risk near expiry.

DEMO: spot **$100**, IV **0.25**, **45 DTE**. Chain mids from module 00.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import strategies, analyzer, payoff, viz

SPOT, VOL, t = 100.0, 0.25, 45/365

In [ ]:
def show(pos):
    s = analyzer.summarize(pos, SPOT, VOL)
    print(s['label'])
    print(f"  net_premium {s['net_premium']:+.0f}  breakevens {[round(b,2) for b in s['breakevens']]}")
    print(f"  max_profit {s['max_profit']:.0f}  max_loss {s['max_loss']:.0f}  POP {s['probability_of_profit']:.2f}")
    g = s['greeks']
    print(f"  delta {g.delta:+.1f}  gamma {g.gamma:+.3f}  theta {g.theta:+.1f}  vega {g.vega:+.1f}")

## 1. The expected move — the ruler

`analyzer.expected_move` gives the 1-sigma range. Everything below is measured against it.

In [ ]:
em = analyzer.expected_move(SPOT, VOL, t)
print(f'45-DTE 1-sigma expected move: +/- {em:.2f}')
print(f'~68% range: {SPOT-em:.1f} to {SPOT+em:.1f}')

## 2. Long vs short straddle (ATM, pure vol)

In [ ]:
long_strad  = strategies.long_straddle((100, 3.91), (100, 3.42), expiry=t)
short_strad = strategies.short_straddle((100, 3.91), (100, 3.42), expiry=t)
show(long_strad); print(); show(short_strad)

The long straddle is **long gamma / long vega / short theta** (needs a move > the 7.33 debit, i.e.
beyond breakevens 92.67/107.33); the short straddle is the exact mirror with **undefined loss**.

## 3. Short strangle (OTM, wider, undefined risk)

In [ ]:
short_strangle = strategies.short_strangle((95, 1.58), (105, 1.85), expiry=t)
show(short_strangle)

Flat profit top between 95 and 105; keeps the 3.43 credit if DEMO stays in range. Compare its
breakevens (91.57/108.43) to the expected move (+/- 8.8) — they sit near the 1-sigma edges.

## 4. Iron condor — the flagship, with the expected move overlaid

Defined-risk short strangle. Plot the payoff and shade the expected-move range so you can see the
profit tent against where the market prices the 1-sigma edges.

In [ ]:
condor = strategies.iron_condor((90, 0.62), (95, 1.58), (105, 1.85), (110, 0.73), expiry=t)
show(condor)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
viz.plot_payoff(condor, spot=SPOT, vol=VOL, ax=ax)
ax.axvspan(SPOT - em, SPOT + em, color='gray', alpha=0.15, label='+/-1 sigma expected move')
ax.set_title('Iron condor payoff vs expected move'); ax.legend()
plt.show()

The profit tent (95-105) sits **inside** the expected-move band, and the breakevens
(92.92/107.08) sit near its edges — a picture of the probability/credit trade-off.

## 5. Iron butterfly and long butterfly (ATM tents)

In [ ]:
iron_fly = strategies.iron_butterfly((90, 0.62), (100, 3.42), (100, 3.91), (110, 0.73), expiry=t)
long_fly = strategies.long_call_butterfly((95, 7.05), (100, 3.91), (105, 1.85), expiry=t)
show(iron_fly); print(); show(long_fly)

The iron fly collects a big credit (**598**) for a narrow tent (breakevens ~94/106); the long fly
is a tiny **108** debit with a ~392 max profit if DEMO pins 100 — a cheap, defined pin bet.

## 6. Gamma risk near expiry — scenario grid on the short strangle

`analyzer.scenario_grid` marks the position to model across price x time. Watch how the P&L swing
per unit of price move *steepens* as days pass (short gamma biting near expiry).

In [ ]:
spots = np.arange(88, 113, 1.0)
grid = analyzer.scenario_grid(short_strangle, spots, days_forward=[0, 20, 40, 44], vol=VOL)
grid.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
viz.plot_pnl_heatmap(grid, ax=ax)
ax.set_title('Short strangle P&L: price x days forward (gamma steepens near expiry)')
plt.show()

Near day 44 the P&L flips from deep red (big loss on a move) to green (full credit if pinned)
over just a few dollars of spot — that knife-edge is short gamma near expiry. It's why these
trades are managed off around 21 DTE.

## Experiments

1. In section 3/4, move the short strikes to the **~16-delta** edges (short 90 put / 110 call).
   How do the credit, POP, and breakevens-vs-expected-move change?
2. Widen the iron condor wings to **85/95/105/115**. More credit, but how much more max loss? Is
   the reward/risk better or worse?
3. In section 5, compare the iron fly's POP to the iron condor's. Which trades a fatter credit for
   a narrower tent, and why?
4. In section 6, rebuild the grid with `vol_shift=[-0.05, 0.0, 0.05]` to add a vol axis. How much
   does a 5-point IV drop help the short strangle (short vega)?
5. Build a **long strangle** and run the same scenario grid. Confirm its heatmap is the mirror —
   it *wants* the big move the short strangle fears.